## DATASET AND SETUP  

### IMPORTS


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from numpy.linalg import slogdet, inv
import matplotlib.pyplot as plt


### -------------------------------
### 1. Load the dataset
### -------------------------------

In [ ]:
digits = load_digits()
X = digits.data      # 64-dimensional feature vectors
y = digits.target    # digit labels (0–9)


### 2. Create 70% / 15% / 15% split
###   using stratified splitting


In [ ]:
# First split: Training (70%) + Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,         # 30% temp set
    stratify=y,             # keep class proportions
    random_state=42
)

# Second split: Validation (15%) + Test (15%)
# 15% is half of the 30% temp set
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,         # half of temp → 15% of original
    stratify=y_temp,
    random_state=42
)

### 3. Standardize the features
###    using only training statistics

In [ ]:
scaler = StandardScaler()

scaler.fit(X_train)      # fit only on training data
X_train_std = scaler.transform(X_train)
X_val_std   = scaler.transform(X_val)
X_test_std  = scaler.transform(X_test)

# Check shapes
print("Training set:", X_train_std.shape)
print("Validation set:", X_val_std.shape)
print("Test set:", X_test_std.shape)

## Gaussian Generative Classifier Class

In [ ]:
class GaussianGenerativeModel:
    def __init__(self, lambda_reg=1e-3):
        """
        lambda_reg : regularisation parameter for Σ
        """
        self.lambda_reg = lambda_reg

    def fit(self, X, y):
        """
        Train the model:
        - Estimate class priors π_k
        - Estimate class means μ_k
        - Estimate shared covariance Σ
        - Apply regularisation Σ + λI
        """
        self.classes = np.unique(y)
        self.K = len(self.classes)
        self.d = X.shape[1]  # dimensionality (64)

        N = X.shape[0]

        # ---- 1. Class Priors π_k ----
        self.pi = np.zeros(self.K)
        for k in self.classes:
            self.pi[k] = np.sum(y == k) / N

        # ---- 2. Class Means μ_k ----
        self.mu = np.zeros((self.K, self.d))
        for k in self.classes:
            self.mu[k] = np.mean(X[y == k], axis=0)

        # ---- 3. Shared Covariance Σ ----
        S = np.zeros((self.d, self.d))
        for i in range(N):
            k = y[i]
            diff = (X[i] - self.mu[k]).reshape(-1, 1)
            S += diff @ diff.T
        S /= N

        # ---- 4. Regularise Σ → Σ + λI ----
        self.Sigma = S + self.lambda_reg * np.eye(self.d)

        # Precompute inverse and log-det for speed
        self.Sigma_inv = inv(self.Sigma)
        sign, logdet = slogdet(self.Sigma)
        self.logdetSigma = logdet

    def _log_gaussian(self, x, k):
        """Compute log N(x ; μ_k, Σ) efficiently."""
        diff = (x - self.mu[k])
        t1 = -0.5 * (diff.T @ self.Sigma_inv @ diff)
        t2 = -0.5 * (self.d * np.log(2 * np.pi) + self.logdetSigma)
        return t1 + t2

    def predict(self, X):
        """
        Predict labels using:
        log π_k + log N(x; μ_k, Σ)
        """
        preds = []
        for x in X:
            scores = []
            for k in self.classes:
                score = np.log(self.pi[k]) + self._log_gaussian(x, k)
                scores.append(score)
            preds.append(np.argmax(scores))
        return np.array(preds)

    def accuracy(self, X, y):
        """Compute accuracy"""
        return np.mean(self.predict(X) == y)


## Train the Model

In [ ]:
lambda_values = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]

best_lambda = None
best_val_acc = -1

for lam in lambda_values:
    model = GaussianGenerativeModel(lambda_reg=lam)
    model.fit(X_train_std, y_train)
    val_acc = model.accuracy(X_val_std, y_val)

    print(f"λ = {lam:.0e}, Validation Accuracy = {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_lambda = lam

print("\nBest λ:", best_lambda)
print("Best validation accuracy:", best_val_acc)


## Train Final Model Using Best λ

In [ ]:
final_model = GaussianGenerativeModel(lambda_reg=best_lambda)
final_model.fit(X_train_std, y_train)

test_acc = final_model.accuracy(X_test_std, y_test)

print("Final Test Accuracy:", test_acc)


### λ values and evaluate on validation set

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

lambda_candidates = [1e-4, 1e-3, 1e-2, 1e-1]

val_results = []

for lam in lambda_candidates:
    model = GaussianGenerativeModel(lambda_reg=lam)
    model.fit(X_train_std, y_train)
    
    val_acc = model.accuracy(X_val_std, y_val)
    val_results.append((lam, val_acc))
    
    print(f"λ = {lam:.0e} → Validation Accuracy = {val_acc:.4f}")

# Select the best λ
best_lambda, best_val_acc = max(val_results, key=lambda x: x[1])

print("\nBest λ:", best_lambda)
print("Best Validation Accuracy:", best_val_acc)


### Combine Training + Validation and Retrain Final Model

In [ ]:
# Combine training and validation sets
X_train_full = np.vstack([X_train_std, X_val_std])
y_train_full = np.concatenate([y_train, y_val])

# Train final model with best lambda
final_model = GaussianGenerativeModel(lambda_reg=best_lambda)
final_model.fit(X_train_full, y_train_full)

# Evaluate on the test set
y_pred_test = final_model.predict(X_test_std)
test_acc = np.mean(y_pred_test == y_test)

print("Final Test Accuracy:", test_acc)


### Compute Precision, Recall, F1 (Macro-Averaged)

In [ ]:
prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, y_pred_test, average='macro'
)

print("Macro Precision:", prec)
print("Macro Recall:", rec)
print("Macro F1-score:", f1)


### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)
cm


In [ ]:

plt.figure(figsize=(8,6))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix (Digits 0–9)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.show()




# **A4. Report Template for Part A (Complete and Polished)**

---

## **1. Explanation of the Generative Model**

### **Model Assumptions**

In this assignment, I used a **Gaussian generative classifier**. The model assumes:

* The **class prior** follows a categorical distribution:
  [
  p(y = k) = \pi_k
  ]
  where (\pi_k) is the probability of observing class (k).

* The **class-conditional distribution** of the features is assumed to be a multivariate Gaussian:
  [
  p(x \mid y = k) = \mathcal{N}(x;,\mu_k,\Sigma)
  ]
  where:

  * (\mu_k) is the mean vector for class (k),
  * (\Sigma) is a **shared covariance matrix** for all classes.

This model assumes that although each digit class has its own mean, all classes share the same spread and correlations between features.

---

### **Parameter Estimation**

All parameters are estimated from the **training set**:

* **Class priors**:
  [
  \pi_k = \frac{\text{number of training samples in class } k}{\text{total number of training samples}}
  ]

* **Class means**:
  For each class, the mean vector is:
  [
  \mu_k = \frac{1}{N_k}\sum_{i: y_i = k} x_i
  ]

* **Shared covariance matrix**:
  Using all training samples:
  [
  \Sigma = \frac{1}{N} \sum_{i=1}^{N} (x_i - \mu_{y_i})(x_i - \mu_{y_i})^\top
  ]

---

### **Why Regularisation Is Needed**

The covariance matrix (\Sigma) can be close to singular because:

* The dataset has **64 dimensions**, but not enough samples per class for a stable estimate.
* Some features are highly correlated.

To stabilise the inverse of (\Sigma), we use:

[
\Sigma_\lambda = \Sigma + \lambda I
]

* If **λ is too small** → covariance is poorly conditioned → overfitting.
* If **λ is too large** → covariance becomes too smooth → underfitting.

Hyperparameter tuning is needed to balance these effects.

---

## **2. Validation Results for Different λ**

| λ value | Validation Accuracy |
| ------- | ------------------- |
| 1e-4    | ...                 |
| 1e-3    | ...                 |
| 1e-2    | ...                 |
| 1e-1    | ...                 |

*(Fill this table with the results from your notebook)*

---

## **3. Final Test Results**

Using the best λ found from validation:

* **Test Accuracy:** …
* **Macro Precision:** …
* **Macro Recall:** …
* **Macro F1-score:** …

### **Confusion Matrix**

(Insert the 10×10 table or image here)

---

## **4. Discussion (1–2 Paragraphs)**

Example structure—you should adapt it with your own wording:

---

### **Digit Confusions**

Some digits are more frequently confused than others. For example, the model commonly mixes up *digit A* with *digit B* (fill in from your confusion matrix). This usually happens when the two digits have similar shapes (e.g., 3 vs 5, 8 vs 9), and the shared covariance assumption cannot fully capture the subtle differences between these classes.

---

### **Effect of λ**

The value of λ had a noticeable effect on performance. When λ was very small, the model tended to overfit, leading to lower validation accuracy. As λ increased, accuracy initially improved because the regularisation stabilised the covariance matrix. However, for very large λ, performance dropped because the covariance became too smooth and class shapes were oversimplified.

---

### **Strengths and Weaknesses of the Gaussian Model**

This Gaussian generative model performs reasonably well on the digits dataset, especially considering its simplicity. It captures the average shape of each digit class. However, assuming a **shared covariance matrix** across all classes is a strong simplification that limits the model’s flexibility. Digits vary significantly in appearance, and a single covariance structure cannot fully describe all classes. As a result, digits that differ subtly or have large intra-class variability are more easily confused.

